# LOCALIZATION SYSTEMS - Fundamentals and Principles of Operation (Practice 1)

To run this practice in Google Colab, use this link: [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mgfernan/gsl_uab/blob/main/plab/p1_fundamentals_trilateration.ipynb)

This lab introduces Time of Arrival (TOA) positioning and iterative estimation under noise.

Learning goals:
- Understand how ranges are computed from geometry.
- Relate range noise to position error.
- Observe how iterative estimation depends on step size and number of iterations.

In [ ]:
# Add some required libraries
import math
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.patches import Circle

from ipywidgets import interactive, fixed
from IPython.display import display

The following helper functions are used later in the lab. No changes are required in this section.

In [2]:
import hashlib

def compute_coordinate(niu: str | int) -> float:
    """
    Computes a coordinate value based on the given NIU using SHA-256 hashing.

    >>> compute_coordinate("1135744")
    1.0
    """

    payload = f"{niu}".encode("utf-8")
    digest = hashlib.sha256(payload).digest()
    value = int.from_bytes(digest[:8], byteorder="big", signed=False)
    return float(((value % 100) + 1.0) / 10.0)

In [ ]:
# ------------------------------------------------------------------------------

def generate_noise_sample(signal: float, snr_dB: float) -> float:

  # sigma2--square of sigma, here we use: SNR_dB = 10log(d^2/sigma^2)
  sigma2 = signal**2 / math.pow(10, snr_dB / 10)

  # We assume additive gaussian noise
  noise = np.random.normal() * math.sqrt(sigma2)

  return float(noise)

# ------------------------------------------------------------------------------

def plot_trilateration(nodes, x: float, y: float,
                       activate_noise: bool = False, snr_db: float = 0.0) -> list[float]:

    _, axes = plt.subplots()

    circles = []
    ranges = []

    for node in nodes:
      # Compute the distance between the node and the receiver
      radius_m = math.sqrt((node[0] - x)**2 + (node[1] - y)**2)

      # Add noise if set
      if activate_noise:
        radius_m += generate_noise_sample(radius_m, snr_db)

      ranges.append(radius_m)

      axes.text(node[0] + 0.5, node[1] + 0.5, f'{radius_m:.2f} m')

      circle = Circle(node, radius=radius_m, fill=False)
      circles.append(circle)

    for circle in circles:
      axes.add_artist(circle)

    axes.axis((-10.0, 20.0, -10.0, 20.0))
    axes.set_aspect("equal")
    axes.plot(nodes[:, 0], nodes[:, 1], 'D', c='red', markersize=10)
    axes.plot(x, y, 'p', c='blue', markersize=10)

    plt.title('Trilateration principle (without noise)')
    plt.show()

    return ranges

# ------------------------------------------------------------------------------

def compute_range(from_position: list[float], to_position: list[float]) -> float:

  return float(np.linalg.norm(np.subtract(to_position, from_position)))

# ------------------------------------------------------------------------------

def compute_gradient(nodes, ranges, rx_position):

  n_nodes = len(nodes)

  # Compute the vector differences between the given position and the nodes
  pos_diff_to_nodes = np.subtract(rx_position, nodes)

  # Compute the norm of each vector difference (i.e. computed ranges)
  range_aprioris = np.linalg.norm(pos_diff_to_nodes, axis=1)

  # Compute the range differences between the incoming (observed) and computed
  # This amount is also called pre-fit residuals
  range_differences = ranges - range_aprioris

  # Compute the partials as division of each component by the range to each node
  partials = pos_diff_to_nodes / range_aprioris.reshape(n_nodes, 1)

  # Compute the gradient components
  gradient_components = range_differences.reshape(n_nodes, 1) * partials
  gradient_x = gradient_components[:, 0]
  gradient_y = gradient_components[:, 1]

  return -2.0 * np.array([sum(gradient_x), sum(gradient_y)])

# ------------------------------------------------------------------------------

def steepest_descent(nodes,
                     ranges,
                     apriori: tuple[float, float] = (0.0, 0.0),
                     step: float = 0.1,
                     num_iterations: int = 30):

    estimations = []

    x = np.array(apriori)

    for _ in range(num_iterations):

      gradient = compute_gradient(nodes, ranges, x)

      x = np.subtract(x, step * gradient)

      estimations.append(x.copy())

    return np.array(estimations)

# ------------------------------------------------------------------------------

## Scenario Definition

The node coordinates used by the TOA positioning method are defined in this section.

In [4]:
# Node coordinates (we create a Numpy Array to take advantage of the cool indexing features)

nodes = np.array([
         [0, 0],
         [0, 10],
         [10, 0]
])

# Number of nodes
n_nodes = len(nodes)

The variable `nodes` is a matrix with one row per node: `[x, y]`.
It is useful for loops and for selecting complete columns or rows

**Hint: NumPy Indexing**

Consider this matrix:

$$
\bf{A} = 
\begin{pmatrix}
1 & 2 & 3 \\
4 & 5 & 6 \\
7 & 8 & 9 \\
\end{pmatrix}
$$

In NumPy, it is written as:

In [5]:
A = np.array(
    [
        [1, 2, 3],
        [4, 5, 6],
        [7, 8, 9],
    ]
)

In [6]:
# Access the first column of A
first_column = A[:, 0]
print("First column of A:", first_column)

# Access the second row of A
second_row = A[1, :]
print("Second row of A:", second_row)

First column of A: [1 4 7]
Second row of A: [4 5 6]


**Exercise 1: Nodes**

Given the matrix `nodes`:
- Store all node x-coordinates in one variable.
- Store all node y-coordinates in one variable.
- Plot node positions with the code below.

In [26]:
# Add code here to extract the x and y values of the nodes into separate arrays

In [ ]:
plt.title('Position of nodes')
plt.xlabel('X coordinate')
plt.ylabel('Y coordinate')
plt.grid()
plt.plot(node_x_values, node_y_values, 'D', c='red', markersize=10)


**Tip**: In `plot`, the first two arguments are x and y values. The style `'D'` plots diamond markers.

**Exercise 2: Rover Coordinates**

Use your NIU(s) with `compute_coordinate` to compute:
- `rover_x_coordinate`
- `rover_y_coordinate`

In this lab, rover coordinates are fixed and are the true rover coordinates used for the rest of the exercises.

In [ ]:
# Add code here to compute the rover coordinates using the compute_coordinate function

In [10]:
true_position = [rover_x_coordinate, rover_y_coordinate]
true_position

[1.4, 9.6]

## Trilateration Principle

This section assumes a noise-free scenario.
With three nodes and their TOA-based distances to the rover, position can be estimated from the intersection of three circles (one circle per node).

The plot below shows this principle using the fixed rover coordinates from NIU hashing.

**Note**: Keep `activate_noise` unchecked for now.

In [ ]:
widget = interactive(plot_trilateration,
                     activate_noise=False, snr_db=(1,30),
                     nodes=fixed(nodes), x=fixed(rover_x_coordinate), y=fixed(rover_y_coordinate))

display(widget)

In [ ]:
# Retrieve the range values from the interactive widget
ranges = widget.result
print(f'Ranges from the widget: {ranges}')


**Exercise 3: Ranges and TOA**

Given a known set of coordinates for the user terminal, compute all the ranges to each node by means of the Euclidian distance with respect to the coordinates of the nodes. Verify that the ranges are consistent with the plot above.

- Write the mathematical expression to compute the distance
- Use Python to compute the ranges yourself. They should match the ones above (`ranges`). 

## Receiver Position Estimation

This section studies position estimation from ranges.
The criterion used is Non-Linear Least Squares (NLS). The method is implemented with an iterative steepest descent algorithm in `steepest_descent`.

**Exercise 4: Steepest Descent**

- Explain the steepest descent idea in simple terms.
- Explain why an a-priori position is required.
- Run the next cells and analyze the plot.
- Did the algorithm converge close to the true position?

In [21]:
# Number of iterations
num_iterations = 30;

# Initial estimation of the position (initial guess also known as apriori)
apriori = (5, 2)

# Mu step
mu_step = 0.1

estimations = steepest_descent(nodes, ranges, apriori, step=mu_step, num_iterations=num_iterations)
final_estimation = estimations[-1]

The next cell plots node positions, true rover position, and estimated position from steepest descent.

In [ ]:
# All solutions estimated by the SD algorithm
plt.plot(estimations[:, 0], estimations[:, 1], '.', color='grey')

# Last solution estimated by the SD algorithm
plt.plot(final_estimation[0], final_estimation[1], 'D', color='black', markersize=10)

# True position
plt.plot(true_position[0], true_position[1], 'D', color='blue', markersize=5)

# Nodes
plt.plot(node_x_values, node_y_values, 'D', c='red', markersize=10)

**Exercise 5: Step Size ($\mu$)**

- What is the role of $\mu$ in steepest descent?
- What happens if $\mu$ is increased by factors of 1000 and 10000?
- What happens if $\mu$ is decreased by factors of 1000 and 10000?

Guidance: observe both convergence speed and stability.

**Exercise 6: Number of Iterations**

- What is the role of the number of iterations?
- What happens when the number is reduced?
- What happens when $\mu$ is very small and the number of iterations is also reduced?

Guidance: relate the result to how far the estimate can move per iteration.

**Exercise 7: Noise**

Now analyze the same scenario with noise enabled.

- What does high SNR mean?
- Go back to the interactive plot and click the `activate_noise` box. Then move the `snr_db` slider.
- What changes when moving `snr_db` to the right or to the left?
- In a low-SNR case, how large is the noise relative to the signal?

**Exercise 8: Low-Noise Scenario (High SNR)**

Set `snr_db` between 20 dB and 30 dB. Run the cells and analyze the effect of:
- Initial position estimate
- Number of iterations
- Step size

How does each parameter affect the final estimate? Justify the answer.

**Important:** Re-run the cell that retrieves `ranges`; otherwise, old values will be reused.

**Exercise 9: High-Noise Scenario (Low SNR)**

Set `snr_db` below 10 dB. How does stronger noise in the measurements affect the final estimate?
Which parameters can be adjusted to compensate for the noise? Justify the answer.

**Important:** Re-run the cell that retrieves `ranges`; otherwise, old values will be reused.